<a href="https://colab.research.google.com/github/PsicoJazz/Analise_de_Dados_Doencas_Cronicas_Marilia-SP/blob/main/dataSUS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Atualiza as ferramentas de empacotamento fundamentais
!pip install --upgrade pip setuptools wheel

# 2. Instala a biblioteca do PySUS e o Pandas para manipulação dos dados
!pip install pysus pandas


  Using cached pysus-2.11.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached Unidecode-1.4.0-py3-none-any.whl.metadata (13 kB)
  Using cached aioftp-0.21.4-py3-none-any.whl.metadata (6.1 kB)
  Using cached bigtree-0.12.5-py3-none-any.whl.metadata (28 kB)
  Using cached boto3-1.43.103-py3-none-any.whl.metadata (6.6 kB)
  Using cached chardet-7.6.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (9.6 kB)
  Using cached dateparser-1.4.3-py3-none-any.whl.metadata (25 kB)
  Using cached dbfread-2.0.7-py2.py3-none-any.whl.metadata (3.3 kB)
  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 101.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 55.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 30.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
# Nova forma de importar: os dados de mortalidade são acessados direto pela raiz ou por subnamespaces dedicados
import pysus

# 1. Definir os códigos IBGE dos municípios da Região de Marília
MAPA_REGIAO = {
    352900: "Marília"
}
CODIGOS_REGIAO = list(MAPA_REGIAO.keys())
ANO = 2024  # Altere para o ano desejado

print(f"Iniciando download dos dados do SIM de SP para o ano {ANO}...")

# 2. Baixar a base do Sistema de Informações sobre Mortalidade (SIM) de São Paulo
# Na nova versão do PySUS, as funções simplificadas retornam um DataFrame pronto do Pandas automaticamente!
df_sp = pysus.ftp.sim(state="SP", year=ANO).to_dataframe()

print("Base de dados carregada do DATASUS! Filtrando dados da região de Marília...")

# 3. Filtrar pelas cidades da região de Marília
df_sp['CODMUNRES'] = pd.to_numeric(df_sp['CODMUNRES'], errors='coerce')
df_regiao = df_sp[df_sp['CODMUNRES'].isin(CODIGOS_REGIAO)].copy()

# Mapeia o nome do município para a nova coluna
df_regiao['NOME_MUNICIPIO'] = df_regiao['CODMUNRES'].map(MAPA_REGIAO)

# 4. Filtrar pelas Doenças Crônicas Não Transmissíveis (CID-10)
# E10-E14: Diabetes | I00-I99: Aparelho Circulatório | J40-J47: Respiratórias Crônicas
def filtrar_cronicas(cid):
    if pd.isna(cid):
        return False
    cid = str(cid).upper()
    return cid.startswith('E1') or cid.startswith('I') or cid.startswith('J4')

# Aplica o filtro usando a causa básica do óbito (CAUSABAS)
df_regiao_cronicas = df_regiao[df_regiao['CAUSABAS'].apply(filtrar_cronicas)]

# 5. Salvar o arquivo finalizado no ambiente do Colab
nome_arquivo = f"doencas_cronicas_REGIAO_marilia_{ANO}.csv"
df_regiao_cronicas.to_csv(nome_arquivo, index=False, encoding='utf-8-sig')

print(f"\n🎉 Mineração concluída!")
print(f"Foram identificados {len(df_regiao_cronicas)} registros de doenças crônicas na Região de Marília em {ANO}.")
print(f"Arquivo disponível na aba de arquivos ao lado: {nome_arquivo}")

PySUS 2.11.2 -- welcome!
Data cache: /root/pysus
Change it with: pysus.set_cache('/your/path')
Browse datasets with: pysus.info()

Iniciando download dos dados do SIM de SP para o ano 2024...


DOSP2024.parquet: 0.00B [00:00, ?B/s]
DOSP2024.parquet:   0%|          | 0.00/20.3M [00:00<?, ?B/s]
DOSP2024.parquet:   0%|          | 65.5k/20.3M [00:00<00:01, 19.9MB/s]
DOSP2024.parquet:   1%|          | 131k/20.3M [00:00<00:14, 1.44MB/s] 
DOSP2024.parquet:   1%|          | 197k/20.3M [00:00<00:14, 1.41MB/s]
DOSP2024.parquet:   1%|▏         | 262k/20.3M [00:00<00:13, 1.47MB/s]
DOSP2024.parquet:   2%|▏         | 328k/20.3M [00:00<00:13, 1.51MB/s]
DOSP2024.parquet:   2%|▏         | 393k/20.3M [00:00<00:12, 1.65MB/s]
DOSP2024.parquet:   2%|▏         | 459k/20.3M [00:00<00:11, 1.77MB/s]
DOSP2024.parquet:   3%|▎         | 524k/20.3M [00:00<00:10, 1.87MB/s]
DOSP2024.parquet:   3%|▎         | 590k/20.3M [00:00<00:09, 2.00MB/s]
DOSP2024.parquet:   3%|▎         | 655k/20.3M [00:00<00:09, 2.11MB/s]
DOSP2024.parquet:   4%|▎         | 721k/20.3M [00:00<00:08, 2.24MB/s]
DOSP2024.parquet:   4%|▍         | 786k/20.3M [00:00<00:08, 2.34MB/s]
DOSP2024.parquet:   4%|▍         | 852k/20.3M [00:00<00:07

Base de dados carregada do DATASUS! Filtrando dados da região de Marília...

🎉 Mineração concluída!
Foram identificados 724 registros de doenças crônicas na Região de Marília em 2024.
Arquivo disponível na aba de arquivos ao lado: doencas_cronicas_REGIAO_marilia_2024.csv


In [ ]:
import pandas as pd
doencasCronicas = pd.read_csv('dcMarilia.csv')
doencasCronicas.head()

,ORIGEM,TIPOBITO,DTOBITO,HORAOBITO,NATURAL,CODMUNNATU,DTNASC,IDADE,SEXO,RACACOR,...,TPRESGINFO,TPNIVELINV,NUDIASINF,DTCADINF,MORTEPARTO,DTCONCASO,FONTESINF,ALTCAUSA,CONTADOR,NOME_MUNICIPIO
0,1,2,2012024,2055.0,835,352900.0,21021942,481,2,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5181,Marília
1,1,2,2012024,1100.0,831,313920.0,3051955,468,1,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5182,Marília
2,1,2,2012024,1624.0,825,250510.0,4091942,481,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5183,Marília
3,1,2,3012024,1750.0,831,315590.0,28061935,488,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8907,Marília
4,1,2,3012024,730.0,835,352710.0,20031932,491,1,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9048,Marília


In [ ]:
doencasCronicas.describe()

In [ ]:
print(doencasCronicas['CAUSABAS'].head())

In [ ]:
print(doencasCronicas.columns.tolist())


In [ ]:
# 1. Criar a lista com os nomes exatos das colunas RELEVANTES que decidimos manter
colunas_relevantes = [
    'CAUSABAS', 'CAUSABAS_O',                           # Dados da Doença (CID-10)
    'LINHAA', 'LINHAB', 'LINHAC', 'LINHAD', 'LINHAII',  # Comorbidades e linhas do atestado
    'CODMUNRES', 'NOME_MUNICIPIO', 'CODMUNOCOR',        # Localização Geográfica
    'LOCOCOR',                                          # Tipo de local (Hospital, Casa, etc.)
    'IDADE', 'SEXO', 'RACACOR', 'ESTCIV', 'ESC2010',    # Perfil Sociodemográfico
    'OCUP',                                             # Ocupação/Trabalho
    'DTOBITO',                                          # Data do evento (Linha do tempo)
    'ASSISTMED'                                         # Teve assistência médica ou não
]

# 2. Filtrar o DataFrame original guardando o resultado em uma nova variável limpa
# O parâmetro errors='ignore' serve para o caso de alguma coluna opcional (como CAUSABAS_O) não existir no ano escolhido
doencasCronicas_limpo = doencasCronicas.filter(items=colunas_relevantes, axis=1)

# 3. Mostrar as primeiras linhas do resultado para conferir a tabela reduzida
print(f"Tabela reduzida com sucesso! Colunas mantidas: {doencasCronicas_limpo.shape[1]}")
doencasCronicas_limpo.head()


In [ ]:
# Dicionário de tradução das colunas do DATASUS
mapa_nomes = {
    'CAUSABAS': 'causa_principal_cid',
    'CAUSABAS_O': 'causa_principal_original_cid',
    'LINHAA': 'comorbidade_linha_a',
    'LINHAB': 'comorbidade_linha_b',
    'LINHAC': 'comorbidade_linha_c',
    'LINHAD': 'comorbidade_linha_d',
    'LINHAII': 'comorbidade_outras_causas',
    'CODMUNRES': 'codigo_municipio_residencia',
    'NOME_MUNICIPIO': 'nome_municipio',
    'CODMUNOCOR': 'codigo_municipio_ocorrencia',
    'LOCOCOR': 'local_ocorrencia_tipo',
    'IDADE': 'idade_bruta',
    'SEXO': 'genero_codigo',
    'RACACOR': 'raca_cor_codigo',
    'ESTCIV': 'estado_civil_codigo',
    'ESC2010': 'escolaridade_codigo',
    'OCUP': 'ocupacao_codigo',
    'DTOBITO': 'data_ocorrencia',
    'ASSISTMED': 'teve_assistencia_medica'
}

# Renomeia as colunas diretamente no DataFrame limpo
doencasCronicas_limpo.rename(columns=mapa_nomes, inplace=True)

# Exibe os novos nomes das colunas para conferir
print("✨ Colunas renomeadas com sucesso!")
print(doencasCronicas_limpo.columns.tolist())


In [ ]:

import numpy as np

# 1. Remover as colunas de comorbidades e códigos brutos que não vão para o dashboard
colunas_para_remover = [
    'comorbidade_linha_a', 'comorbidade_linha_b', 'comorbidade_linha_c',
    'comorbidade_linha_d', 'comorbidade_outras_causas', 'causa_principal_original_cid',
    'codigo_municipio_residencia', 'codigo_municipio_ocorrencia'
]
# Usamos errors='ignore' caso alguma dessas já tenha sido removida antes
df_dash = doencasCronicas_limpo.drop(columns=colunas_para_remover, errors='ignore').copy()


# 2. TRATAMENTO DA IDADE (Correção do padrão DATASUS)
# No SUS: 400 a 499 significa anos (ex: 465 = 65 anos). Valores menores são meses/dias (convertemos para 0 anos).
def corrigir_idade(idade_bruta):
    try:
        id_num = int(float(idade_bruta))
        if id_num >= 400 and id_num <= 499:
            return id_num - 400
        elif id_num >= 500: # Centenários (ex: 502 = 102 anos)
            return id_num - 400
        else:
            return 0 # Menores de 1 ano (meses/dias)
    except:
        return np.nan

df_dash['idade'] = df_dash['idade_bruta'].apply(corrigir_idade)
df_dash.drop(columns=['idade_bruta'], inplace=True) # Remove a coluna confusa antiga


# 3. TRATAMENTO DO GÊNERO/SEXO
mapa_sexo = {1: 'Masculino', 2: 'Feminino', '1': 'Masculino', '2': 'Feminino'}
df_dash['genero'] = df_dash['genero_codigo'].map(mapa_sexo).fillna('Não Informado')
df_dash.drop(columns=['genero_codigo'], inplace=True)


# 4. TRATAMENTO DA RAÇA / COR
mapa_raca = {
    1: 'Branca', 2: 'Preta', 3: 'Amarela', 4: 'Parda', 5: 'Indígena',
    '1': 'Branca', '2': 'Preta', '3': 'Amarela', '4': 'Parda', '5': 'Indígena'
}
df_dash['raca_cor'] = df_dash['raca_cor_codigo'].map(mapa_raca).fillna('Não Informado')
df_dash.drop(columns=['raca_cor_codigo'], inplace=True)


# 5. TRATAMENTO DO LOCAL DE OCORRÊNCIA
mapa_local = {
    1: 'Hospital', 2: 'Outro Estab. Saúde', 3: 'Domicílio', 4: 'Via Pública', 5: 'Outros',
    '1': 'Hospital', '2': 'Outro Estab. Saúde', '3': 'Domicílio', '4': 'Via Pública', '5': 'Outros'
}
df_dash['local_ocorrencia'] = df_dash['local_ocorrencia_tipo'].map(mapa_local).fillna('Não Informado')
df_dash.drop(columns=['local_ocorrencia_tipo'], inplace=True)


# 6. TRATAMENTO DA ASSISTÊNCIA MÉDICA
mapa_assist = {1: 'Sim', 2: 'Não', '1': 'Sim', '2': 'Não'}
df_dash['teve_assistencia_medica'] = df_dash['teve_assistencia_medica'].map(mapa_assist).fillna('Não Informado')


# 7. TRATAMENTO E AGRUPAMENTO DOS CIDs (Dar nome real às doenças crônicas)
def traduzir_cid_grupo(cid):
    if pd.isna(cid):
        return 'Outras / Não Informado'
    cid = str(cid).upper().strip()

    if cid.startswith('E10') or cid.startswith('E11') or cid.startswith('E12') or cid.startswith('E13') or cid.startswith('E14'):
        return 'Diabetes Mellitus'
    elif cid.startswith('I10') or cid.startswith('I11') or cid.startswith('I12') or cid.startswith('I13') or cid.startswith('I15'):
        return 'Doenças Hipertensivas'
    elif cid.startswith('I20') or cid.startswith('I21') or cid.startswith('I22') or cid.startswith('I23') or cid.startswith('I24') or cid.startswith('I25'):
        return 'Doenças Isquêmicas do Coração (Infarto/Angina)'
    elif cid.startswith('I60') or cid.startswith('I61') or cid.startswith('I62') or cid.startswith('I63') or cid.startswith('I64') or cid.startswith('I69'):
        return 'Doenças Cerebrovasculares (AVC/Derrame)'
    elif cid.startswith('J40') or cid.startswith('J41') or cid.startswith('J42') or cid.startswith('J43') or cid.startswith('J44'):
        return 'Doenças Respiratórias Crônicas (DPOC/Bronquite/Enfisema)'
    else:
        return 'Outras Doenças Crônicas Cardiovasculares/Metabólicas'

df_dash['doenca_cronica_grupo'] = df_dash['causa_principal_cid'].apply(traduzir_cid_grupo)


# 8. AJUSTAR FORMATO DA DATA
df_dash['data_ocorrencia'] = pd.to_datetime(df_dash['data_ocorrencia'], format='%d%m%Y', errors='coerce')


# 9. REORGANIZAR AS COLUNAS FINAIS (Manter uma ordem limpa)
colunas_finais = [
    'data_ocorrencia', 'nome_municipio', 'doenca_cronica_grupo',
    'causa_principal_cid', 'idade', 'genero', 'raca_cor',
    'local_ocorrencia', 'teve_assistencia_medica'
]
df_dashboard_pronto = df_dash[colunas_finais].copy()

# Salvar o arquivo final perfeito para o Dashboard
df_dashboard_pronto.to_csv('dados_mortalidade_marilia_dashboard.csv', index=False, encoding='utf-8-sig')

print("🚀 Base de dados higienizada e PRONTA para o Dashboard!")
print(f"Colunas resultantes: {df_dashboard_pronto.columns.tolist()}")
df_dashboard_pronto.head()


In [ ]:
# Exporta o DataFrame pronto para um arquivo do Excel (.xlsx)
# index=False evita que o Pandas crie uma coluna extra de numeração (0, 1, 2...) no Excel
df_dashboard_pronto.to_excel('dados_marilia_dashboard.xlsx', index=False)

print("🎉 Arquivo do Excel gerado com sucesso!")
